# Cleaning 2 — PCA for PODES (Paper 1 Equity & Paper 2 Health Infrastructure)

Load DATA_BERSIH gpkg files by year and run PCA using the variable lists for:
- **Paper 1 (Equity)**: PCA_P1_2011, PCA_P1_2014, … PCA_P1_2024
- **Paper 2 (Health Infrastructure Access)**: PCA_P2_2011, … PCA_P2_2024

Categorical variables are converted to dummies (or kept numeric where ordinal) so PCA does not break.

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

BASE = Path("/Users/athamawardi/Desktop/Research-Projects/KRE_Equity")
DATA_BERSIH = Path("/Users/athamawardi/Desktop/Research-Projects/KRE_Equity/Data/DATA_BERSIH")

SAVE_CSV = True

## PCA variable lists by PODES wave

Paper 1 (Equity) and Paper 2 (Health Infrastructure Access).

In [2]:
# PAPER 1 (EQUITY) — PCA input lists by year
PCA_P1_2011 = """
R711A R711B R711C R1402_COL5 R501A R502A R502B R503 R504 R505A R505B R506 R507 R508A R508B R509A R509B
R510A R510B1 R510B2 R511A R511B R511C R512A R512B R1001A R1001B1 R1005A R1005B R1007A R1007B R701AK2
R701AK3 R701AK4 R701BK2 R701BK3 R701BK4 R701CK2 R701CK3 R701CK4 R701DK2 R701DK3 R701DK4 R701EK2
R701EK3 R701EK4 R701FK2 R701FK3 R701FK4 R701GK2 R701GK3 R701GK4 R701HK2 R701HK3 R701HK4 R701IK2
R701IK3 R701IK4 R704AK2 R704BK2 R704CK2 R704DK2 R704EK2 R704FK2 R704GK2 R704HK2 R704IK2 R704JK2
R704KK2 R704LK2 R704AK3 R704BK3 R704CK3 R704DK3 R704EK3 R704FK3 R704GK3 R704HK3 R704IK3 R704JK3
R704KK3 R704LK3 R704AK4 R704BK4 R704CK4 R704DK4 R704EK4 R704FK4 R704GK4 R704HK4 R704IK4 R704JK4
R704KK4 R704LK4 R1201A R1201B R1201C R1202A R1202B R1202C R1203A R1203B R1203C R1204A R1204B
R1204C R1205A R1205B R1205C R1206 R1207 R1208 R1209 R1210 R1211 R1212 R1213A R1213B R1213C R1213D
R1214B R1215A_K2 R1215A_K3 R1215A_K4 R1215B_K2 R1215B_K3 R1215B_K4
""".split()

PCA_P1_2014 = """
R711A R711B R711C R1401A3_K2 R501A1 R501A2 R501B R502A R502B R503A1 R503A2 R503A3 R503A4 R503A5 R503B
R504 R507A R507B R508A R508B2_K2 R509A_K2 R509A_K3 R509A_K4 R510A R510B1K2 R510B1K3 R510B1K4 R510B2K2
R510B2K3 R510B2K4 R510D1 R512A_K2 R512A_K3 R512A_K4 R1005A R1005B R1007A R1007B R701A_K2 R701A_K3
R701A_K4 R701B_K2 R701B_K3 R701B_K4 R701C_K2 R701C_K3 R701C_K4 R701D_K2 R701D_K3 R701D_K4 R701E_K2
R701E_K3 R701E_K4 R701F_K2 R701F_K3 R701F_K4 R701G_K2 R701G_K3 R701G_K4 R701H_K2 R701H_K3 R701H_K4
R701I_K2 R701I_K3 R701I_K4 R704A_K2 R704B_K2 R704C_K2 R704D_K2 R704E_K2 R704F_K2 R704G_K2 R704H_K2
R704I_K2 R704J_K2 R704K_K2 R704L_K2 R704A_K3 R704B_K3 R704C_K3 R704D_K3 R704E_K3 R704F_K3 R704G_K3
R704H_K3 R704I_K3 R704J_K3 R704K_K3 R704L_K3 R704A_K4 R704B_K4 R704C_K4 R704D_K4 R704E_K4 R704F_K4
R704G_K4 R704H_K4 R704I_K4 R704J_K4 R704K_K4 R704L_K4 R1201A R1201B R1201C R1202A R1202B R1202C
R1203A R1203B R1203C R1204A R1204B R1204C R1205A R1205B R1205C R1206 R1207 R1208 R1209 R1210 R1211
R1212 R1213A R1213B R1213C R1213D
""".split()

PCA_P1_2018 = """
R711A R711B R711C R1401A3_K2 R501A1 R501A2 R501B R502A R502B R503A1 R503A2 R503A3 R503A4 R503A5 R503B
R504 R507A R507B R508A R508B2_K2 R509D1 R510A R510B1K2 R510B1K3 R510B1K4 R510B2K2 R510B2K3 R510B2K4
R510D1 R512A_K2 R512A_K3 R512A_K4 R1001A R1001B1 R1005A R1005B R1005C R1005D R1007A R1007B R701AK2
R701AK3 R701AK4 R701BK2 R701BK3 R701BK4 R701CK2 R701CK3 R701CK4 R701DK2 R701DK3 R701DK4 R701EK2
R701EK3 R701EK4 R701FK2 R701FK3 R701FK4 R701GK2 R701GK3 R701GK4 R701HK2 R701HK3 R701HK4 R701IK2
R701IK3 R701IK4 R704AK2 R704BK2 R704CK2 R704DK2 R704EK2 R704FK2 R704GK2 R704HK2 R704IK2 R704JK2
R704KK2 R704LK2 R704MK2 R704AK3 R704BK3 R704CK3 R704DK3 R704EK3 R704FK3 R704GK3 R704HK3 R704IK3
R704JK3 R704KK3 R704LK3 R704MK3 R704AK4 R704BK4 R704CK4 R704DK4 R704EK4 R704FK4 R704GK4 R704HK4
R704IK4 R704JK4 R704KK4 R704LK4 R704MK4 R1201A R1201B R1201C R1202A R1202B R1202C R1203A R1203B
R1203C R1204A R1204B R1204C R1205A R1205B R1205C R1206 R1207 R1208 R1209 R1210 R1211 R1212 R1213A
R1213B R1213C R1213D
""".split()

PCA_P1_2019 = """
R711A R711B R711C R1401A3_K2 R401A1 R401A2 R401B R402A R402B R403 R404 R1301 R501A1 R501A2 R501B R503A1
R503A2 R503A3 R503A4 R503A5 R503B R504 R507A R507B R508A R508B2_K2 R509D1 R510A R510B1K2 R510B1K3
R510B1K4 R510B2K2 R510B2K3 R510B2K4 R510D1 R512A_K2 R512A_K3 R512A_K4 R1001A R1001B1 R1004 R1005A
R1005B R1005C R1005D R1202A R1202B R1202C R1203A R1203B R1203C R1204A R1204B R1204C R1205A1 R1205A2
R1205A3 R1205A4 R1205A5 R1205B1 R1205B2 R1205B3 R1205B4 R1205B5 R1205C1 R1205C2 R1205C3 R1205C4
R1205C5 R1206A1 R1206A2 R1206A3 R1206B1 R1206B2 R1206B3 R1207A1 R1207A2 R1207B1 R1207B2 R1207C1
R1207C2 R1208A1 R1208A2 R1208A3 R1208B1 R1208B2 R1208B3 R1208C1 R1208C2 R1208C3 R1209A1 R1209A2
R1209A3 R1209B1 R1209B2 R1209B3 R1209C1 R1209C2 R1209C3 R1210A1 R1210A2 R1210A3 R1210B1 R1210B2
R1210B3 R1210C1 R1210C2 R1210C3 R1211A1 R1211A2 R1211A3 R1211B1 R1211B2 R1211B3 R1211C1 R1211C2
R1211C3 R1212A1 R1212A2 R1212A3 R1212B1 R1212B2 R1212B3 R1212C1 R1212C2 R1212C3 R1213A R1213B
R1213C R1213D R701AK2 R701AK3 R701AK4 R701BK2 R701BK3 R701BK4 R701CK2 R701CK3 R701CK4 R701DK2
R701DK3 R701DK4 R701EK2 R701EK3 R701EK4 R701FK2 R701FK3 R701FK4 R701GK2 R701GK3 R701GK4 R701HK2
R701HK3 R701HK4 R701IK2 R701IK3 R701IK4 R704AK2 R704BK2 R704CK2 R704DK2 R704EK2 R704FK2 R704GK2
R704HK2 R704IK2 R704JK2 R704KK2 R704LK2 R704MK2 R705A R705B R705C
""".split()

PCA_P1_2020 = """
R501A1 R501A2 R501B R502A R502B R503A1 R503A2 R503A3 R503A4 R503A5 R503A6 R503A7 R503A8 R503A9 R503A10
R503B R507A R507B R508 R509 R510A R510B1K2 R510B1K3 R510B1K4 R510B2K2 R510B2K3 R510B2K4 R512A_K2
R512A_K3 R512A_K4 R1401B3K2 R1001A R1001B1 R1202A R1202B R1202C R1203A R1203B R1203C R1204A R1204B
R1204C R1205A1 R1205A2 R1205A3 R1205A4 R1205A5 R1205B1 R1205B2 R1205B3 R1205B4 R1205B5 R1205C1
R1205C2 R1205C3 R1205C4 R1205C5 R1206A1 R1206A2 R1206A3 R1206B1 R1206B2 R1206B3 R1207A1 R1207A2
R1207B1 R1207B2 R1207C1 R1207C2 R1208A1 R1208A2 R1208A3 R1208B1 R1208B2 R1208B3 R1208C1 R1208C2
R1208C3 R1209A1 R1209A2 R1209A3 R1209B1 R1209B2 R1209B3 R1209C1 R1209C2 R1209C3 R1210A1 R1210A2
R1210A3 R1210B1 R1210B2 R1210B3 R1210C1 R1210C2 R1210C3 R1211A1 R1211A2 R1211A3 R1211B1 R1211B2
R1211B3 R1211C1 R1211C2 R1211C3 R1212A1 R1212A2 R1212A3 R1212B1 R1212B2 R1212B3 R1212C1 R1212C2
R1212C3 R1213A R1213B R1213C R1213D R903AK2 R903AK3 R903AK4 R903BK2 R903BK3 R903BK4 R903CK2 R903CK3
R903CK4 R903DK2 R903DK3 R903DK4 R906AK4 R906BK4 R907AK2 R907AK3 R907AK4 R907BK2 R907BK3 R907BK4
R907CK2 R907CK3 R907CK4 R907DK2 R907DK3 R907DK4 R907EK2 R907EK3 R907EK4 R907FK2 R907FK3 R907FK4
R907GK2 R907GK3 R907GK4 R907HK2 R907HK3 R907HK4 R906EK2 R906EK3 R906EK4 R902BK2 R902BK3 R902BK4
R902CK2 R902CK3 R902CK4 R902DK2 R902DK3 R902DK4 R902EK2 R902EK3 R902EK4 R902FK2 R902FK3 R902FK4
R701BK2 R701BK3 R701BK4 R705A R705B R705C R704AK2 R704BK2 R704CK2 R704DK2 R704EK2 R704FK2 R704GK2
R704HK2 R704IK2 R704JK2 R704KK2 R704LK2 R704MK2 R704AK3 R704BK3 R704CK3 R704DK3 R704EK3 R704FK3
R704GK3 R704HK3 R704IK3 R704JK3 R704KK3 R704LK3 R704MK3 R704AK4 R704BK4 R704CK4 R704DK4 R704EK4
R704FK4 R704GK4 R704HK4 R704IK4 R704JK4 R704KK4 R704LK4 R704MK4 R711A R711B R711C
""".split()

PCA_P1_2021 = """
R1501A_K2 R1501A_K3 R1501A_K4 R1501B_K2 R1501B_K3 R1501B_K4 R1502_5 R1502_6 R1502_7 R1502_8 R1502_9
R1502_10 R1502_11 R1502_1A R1502_1B R1502_1C R1502_1D R501A1 R501A2 R501B R502A R502B R502C R503A1
R503A2 R503A3 R503A4 R503A5 R503A6 R503A7 R503A8 R503A9 R503A10 R503B R507A R507B R508 R509 R510A
R510B1K2 R510B1K3 R510B1K4 R510B2K2 R510B2K3 R510B2K4 R512A_K2 R512A_K3 R512A_K4 R1401B3K2 R1001A
R1001B1 R1001C2 R1001C3 R1001C4 R1001D1 R1001D2 R1001D3 R1202A R1202B R1202C R1203A R1203B R1203C
R1204A R1204B R1204C R1205A1 R1205A2 R1205A3 R1205A4 R1205A5 R1205B1 R1205B2 R1205B3 R1205B4 R1205B5
R1205C1 R1205C2 R1205C3 R1205C4 R1205C5 R1206A1 R1206A2 R1206A3 R1206B1 R1206B2 R1206B3 R1207A1
R1207A2 R1207B1 R1207B2 R1207C1 R1207C2 R1208A1 R1208A2 R1208A3 R1208B1 R1208B2 R1208B3 R1208C1
R1208C2 R1208C3 R1209A1 R1209A2 R1209A3 R1209B1 R1209B2 R1209B3 R1209C1 R1209C2 R1209C3 R1210A1
R1210A2 R1210A3 R1210B1 R1210B2 R1210B3 R1210C1 R1210C2 R1210C3 R1211A1 R1211A2 R1211A3 R1211B1
R1211B2 R1211B3 R1211C1 R1211C2 R1211C3 R1212A1 R1212A2 R1212A3 R1212B1 R1212B2 R1212B3 R1212C1
R1212C2 R1212C3 R1213A R1213B R1213C R1213D R701AK2 R701AK3 R701AK4 R701BK2 R701BK3 R701BK4 R701CK2
R701CK3 R701CK4 R701DK2 R701DK3 R701DK4 R701EK2 R701EK3 R701EK4 R701FK2 R701FK3 R701FK4 R701GK2 R701GK3
R701GK4 R701HK2 R701HK3 R701HK4 R701IK2 R701IK3 R701IK4 R704AK2 R704BK2 R704CK2 R704DK2 R704EK2 R704FK2
R704GK2 R704HK2 R704IK2 R704JK2 R704KK2 R704LK2 R704MK2 R705A R705B R705C R1004 R1005A R1005B R1005C
R1005D
""".split()

PCA_P1_2024 = """
R1501A_K2 R1501A_K3 R1501A_K4 R1501B_K2 R1501B_K3 R1501B_K4 R1502_5 R1502_6 R1502_7 R1502_8 R1502_9
R1502_10 R1502_11 R1502_1A R1502_1B R1502_1C R1502_1D R501A1 R501A2 R501B R502 R503A1 R503A2 R503A3
R503A4 R503A5 R503A6 R503A7 R503A8 R503A9 R503A10 R503B R507A R507B R508 R509 R510A R510B1K2 R510B1K3
R510B1K4 R510B2K2 R510B2K3 R510B2K4 R512A_K2 R512A_K3 R512A_K4 R1401B3K2 R1001A R1001B1 R1202A R1202B
R1202C R1203A R1203B R1203C R1204A R1204B R1204C R1205A1 R1205A2 R1205A3 R1205A4 R1205A5 R1205B1 R1205B2
R1205B3 R1205B4 R1205B5 R1205C1 R1205C2 R1205C3 R1205C4 R1205C5 R1206A1 R1206A2 R1206A3 R1206B1 R1206B2
R1206B3 R1207A1 R1207A2 R1207B1 R1207B2 R1207C1 R1207C2 R1208A1 R1208A2 R1208A3 R1208B1 R1208B2 R1208B3
R1208C1 R1208C2 R1208C3 R1209A1 R1209A2 R1209A3 R1209B1 R1209B2 R1209B3 R1209C1 R1209C2 R1209C3 R1210A1
R1210A2 R1210A3 R1210B1 R1210B2 R1210B3 R1210C1 R1210C2 R1210C3 R1211A1 R1211A2 R1211A3 R1211B1 R1211B2
R1211B3 R1211C1 R1211C2 R1211C3 R1212A1 R1212A2 R1212A3 R1212B1 R1212B2 R1212B3 R1212C1 R1212C2 R1212C3
R1213A R1213B R1213C R1213D R701AK2 R701AK3 R701AK4 R701BK2 R701BK3 R701BK4 R701CK2 R701CK3 R701CK4 R701DK2
R701DK3 R701DK4 R701EK2 R701EK3 R701EK4 R701FK2 R701FK3 R701FK4 R701GK2 R701GK3 R701GK4 R701HK2 R701HK3 R701HK4
R701IK2 R701IK3 R701IK4 R704AK2 R704BK2 R704CK2 R704DK2 R704EK2 R704FK2 R704GK2 R704HK2 R704IK2 R704JK2 R704KK2
R704LK2 R704MK2 R705A R705B R705C R1004 R1005A R1005B R1005C R1005D R1007A R1007B R805C R805D R805E R601AK2 R601AK3
R601AK4 R601BK2 R601BK3 R601BK4 R601CK2 R601CK3 R601CK4 R601DK2 R601DK3 R601DK4 R601EK2 R601EK3 R601EK4 R601FK2 R601FK3
R601FK4 R601GK2 R601GK3 R601GK4 R601HK2 R601HK3 R601HK4 R907E_K4 R907F_K4 R907G_K4 R907H_K4 R908E_K4 R908F_K4 R908G_K4
R908H_K4 R903AK2 R903AK3 R903AK4 R903BK2 R903BK3 R903BK4 R903CK2 R903CK3 R903CK4 R903DK2 R903DK3 R903DK4 R804AK2 R804AK3
R804AK4 R804BK2 R804BK3 R804BK4 R804CK2 R804CK3 R804CK4 R804DK2 R804DK3 R804DK4 R805AK2 R805AK3 R805AK4 R805BK2 R805BK3
R805BK4 R904E_K2 R904E_K3 R904E_K4 R904F_K2 R904F_K3 R904F_K4 R904G_K2 R904G_K3 R904G_K4 R904H_K2 R904H_K3 R904H_K4 R905A1
R905A2 R905B R906A1 R906A2 R906B
""".split()

P1_LISTS = {
    2011: [x.strip() for x in PCA_P1_2011 if x.strip()],
    2014: [x.strip() for x in PCA_P1_2014 if x.strip()],
    2018: [x.strip() for x in PCA_P1_2018 if x.strip()],
    2019: [x.strip() for x in PCA_P1_2019 if x.strip()],
    2020: [x.strip() for x in PCA_P1_2020 if x.strip()],
    2021: [x.strip() for x in PCA_P1_2021 if x.strip()],
    2024: [x.strip() for x in PCA_P1_2024 if x.strip()],
}

In [3]:
# PAPER 2 (HEALTH INFRASTRUCTURE ACCESS) — PCA input lists by year
PCA_P2_2011 = """
R704AK4 R704CK4 R704DK4 R704EK4 R704FK4 R704GK4 R704HK4 R704IK4 R704KK4 R704LK4 R1004AK3 R1004AK4
R1004BK3 R1004BK4 R1004CK3 R1004CK4 R704AK3 R704CK3 R704DK3 R704EK3 R704HK3 R704IK3 R704JK3 R705A
R705B R705C R706AK2 R706AK3 R706AK4 R706BK2 R704KK2 R704KK3 R704KK5 R704LK2 R704LK5 R704FK3 R704GK3
R707A1 R707A2 R707B R707C R707E R711 R705 R511AK2 R511AK3
""".split()

PCA_P2_2014 = """
R704A_K5 R704B_K5 R704C_K5 R704D_K5 R704E_K5 R704F_K5 R704G_K5 R704H_K5 R704I_K5 R704J_K5 R704L_K5
R704A_K4 R704B_K4 R704C_K4 R704D_K4 R704E_K4 R704F_K4 R704G_K4 R704H_K4 R704I_K4 R704J_K4 R704L_K4
R1001C1 R1001C2 R1001C3 R1002A_K5 R1002A_K6 R1002B_K5 R1002B_K6 R1002C_K5 R1002C_K6 R1002D_K5
R1002D_K6 R704A_K3 R704B_K3 R704C_K3 R704D_K3 R704E_K3 R704F_K3 R704I_K3 R704K_K3 R704L_K2 R704L_K3
R704M_K2 R704M_K4 R704M_K5 R704G_K3 R704H_K3 R706 R711A R711C R705 R506 R509D1 R512A1_K3 R512A_K4
""".split()

PCA_P2_2018 = """
R704AK4 R704BK4 R704CK4 R704DK4 R704EK4 R704FK4 R704GK4 R704IK4 R704JK4 R704KK4 R704LK4 R704AK3 R704BK3
R704CK3 R704DK3 R704EK3 R704FK3 R704GK3 R704IK3 R704JK3 R704KK3 R704LK3 R1001C1 R1001C2 R1001C3
R1002AK3 R1002AK4 R1002BK3 R1002BK4 R1002CK3 R1002CK4 R1002DK3 R1002DK4 R704AK2 R704BK2 R704CK2
R704DK2 R704EK2 R704FK2 R704JK2 R705A R705B R704LK2 R704MK2 R704MK3 R704MK4 R704GK2 R704IK2 R706
R711A R705 R506 R510D1 R513AK3 R513AK4
""".split()

PCA_P2_2019 = """
R502AK4 R502BK4 R502CK4 R502DK4 R502EK4 R502FK4 R502GK4 R502IK4 R502JK4 R502KK4 R502LK4 R502AK3 R502BK3
R502CK3 R502DK3 R502EK3 R502FK3 R502GK3 R502IK3 R502JK3 R502KK3 R502LK3 R701C1 R701C2 R701C3 R702AK3
R702AK4 R702BK3 R702BK4 R502AK2 R502BK2 R502CK2 R502DK2 R502EK2 R502FK2 R502JK2 R502KK2 R502N R502N1
R502N2 R502LK2 R502MK2 R502MK3 R502MK4 R502GK2 R502IK2 R502
""".split()

PCA_P2_2020 = """
R603AK4 R603BK4 R603CK4 R603DK4 R603EK4 R603FK4 R603GK4 R603IK4 R603JK4 R603KK4 R603LK4 R603AK3 R603BK3
R603CK3 R603DK3 R603EK3 R603FK3 R603GK3 R603HK3 R603IK3 R603JK3 R603KK3 R603LK3 R801C1 R801C2 R801C3
R802AK3 R802AK4 R802BK3 R802BK4 R603AK2 R603BK2 R603CK2 R603DK2 R603EK2 R603FK2 R603JK2 R603LK2
R603MK2 R603MK3 R603MK4 R603GK2 R603IK2 R603N3K2 R603N4K2 R603NK2
""".split()

PCA_P2_2021 = """
R704AK7 R704BK7 R704CK7 R704DK7 R704EK7 R704FK7 R704GK7 R704IK7 R704JK7 R704KK7 R704LK7 R704AK3 R704BK3
R704CK3 R704DK3 R704EK3 R704FK3 R704GK3 R704IK3 R704JK3 R704KK3 R704LK3 R704AK5 R704BK5 R704CK5
R704DK5 R704EK5 R704FK5 R704GK5 R704HK5 R704IK5 R704JK5 R704KK5 R704LK5 R704MK5 R1001C1 R1001C2
R1001C3 R1002AK3 R1002AK4 R1002BK3 R1002BK4 R1002CK3 R1002CK4 R1002DK3 R1002DK4 R1207AK5 R1207BK5
R1207CK5 R1207DK5 R1207EK5 R1207FK5 R1207GK5 R1207HK5 R1207IK5 R1207JK5 R1209AK5 R1209BK5 R1209CK5
R1209DK5 R1209EK5 R1209FK5 R1209GK5 R704AK2 R704BK2 R704CK2 R704DK2 R704EK2 R704FK2 R704JK2 R705A
R705B R705C R704LK2 R704LK6 R704MK2 R704MK3 R704MK6 R704MK7 R706 R705 R710 R1502_1A R1502_1B R1502_1C
R1502_1D R1502_2 R1502_3 R1502_4 R1502_5 R1502_6 R1502_7 R1502_8 R1502_9 R1502_10 R1502_11 R506
R510C1 R510C2A R510C2B R510C2C R513AK3 R513AK4 R907E_K4 R907F_K4 R907G_K4 R907H_K4 R908E_K4 R908F_K4
R908G_K4 R908H_K4 R1013H_K3 R1013I_K3 R1013J_K3
""".split()

PCA_P2_2024 = """
R704AK4 R704BK4 R704CK4 R704DK4 R704EK4 R704FK4 R704GK4 R704IK4 R704JK4 R704KK4 R704LK4 R704AK3 R704BK3
R704CK3 R704DK3 R704EK3 R704FK3 R704GK3 R704IK3 R704JK3 R704KK3 R704LK3 R1001C1 R1001C2 R1001C3
R1002AK3 R1002AK4 R1002BK3 R1002BK4 R1002CK3 R1002CK4 R1002DK3 R1002DK4 R704AK2 R704BK2 R704CK2
R704DK2 R704EK2 R704FK2 R704JK2 R705A R705B R705C R704LK2 R704MK2 R704MK3 R704MK4 R706 R705 R1502_1
R1502_1A R1502_1B R1502_1C R1502_1D R1502_2 R1502_3 R1502_4 R1502_5 R1502_6 R1502_7 R1502_8 R1502_9
R1502_10 R1502_11 R709 R507 R511C1 R511C2A R511C2B R511C2C R514A_K3 R514A_K4 R907E_K4 R907F_K4
R907G_K4 R907H_K4 R908E_K4 R908F_K4 R908G_K4 R908H_K4 R1013H_K3 R1013I_K3 R1013J_K3
""".split()

P2_LISTS = {
    2011: [x.strip() for x in PCA_P2_2011 if x.strip()],
    2014: [x.strip() for x in PCA_P2_2014 if x.strip()],
    2018: [x.strip() for x in PCA_P2_2018 if x.strip()],
    2019: [x.strip() for x in PCA_P2_2019 if x.strip()],
    2020: [x.strip() for x in PCA_P2_2020 if x.strip()],
    2021: [x.strip() for x in PCA_P2_2021 if x.strip()],
    2024: [x.strip() for x in PCA_P2_2024 if x.strip()],
}

## Helper: resolve variable names and build PCA matrix

PODES column names may use underscore (e.g. `R704A_K2`) or no underscore (`R704AK2`). We try both. Categorical columns are converted to dummies.


In [4]:
def resolve_pca_columns(df, requested_vars):
    cols = set(df.columns)
    resolved = []
    for v in requested_vars:
        v = v.strip()
        if not v:
            continue
        if v in cols:
            resolved.append(v)
            continue
        alt = v.replace("_", "")
        if alt in cols:
            resolved.append(alt)
            continue
        for i in range(len(v) - 1, 0, -1):
            if v[i] in "23456789" and v[i - 1] in "ABCDEFGHIJKLMN":
                alt2 = v[:i] + "_" + v[i:]
                if alt2 in cols:
                    resolved.append(alt2)
                    break
                break
        else:
            if v not in [x for x in resolved]:
                pass  # skip missing
    return list(dict.fromkeys(resolved))


def prepare_pca_matrix(df, pca_cols, min_obs=50):
    if not pca_cols:
        return None, None
    sub = df[pca_cols].copy()
    for c in sub.columns:
        if sub[c].dtype == object or sub[c].dtype.name == "category":
            sub[c] = pd.to_numeric(sub[c], errors="coerce")
    sub = sub.loc[:, sub.notna().any(axis=0)]
    if sub.shape[1] < 2:
        return None, None
    valid = sub.notna().any(axis=1)
    sub = sub.loc[valid]
    if len(sub) < min_obs:
        return None, None
    for c in sub.columns:
        sub[c] = sub[c].fillna(sub[c].median())
    return sub, sub.index


def run_pca(X, n_components=1, standardize=True):
    if X is None or len(X) < 2 or X.shape[1] < 2:
        return None, None, None
    n_components = min(n_components, X.shape[0], X.shape[1])
    if standardize:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
    else:
        X_scaled = X.values
    pca = PCA(n_components=n_components)
    scores = pca.fit_transform(X_scaled)
    return pca, scores, pca.explained_variance_ratio_

gpkg_2011 = DATA_BERSIH / "adm4_podes_2011_with_iup_flag.gpkg"
if gpkg_2011.exists():
    gdf_test = gpd.read_file(gpkg_2011, rows=100)
    p1_cols = resolve_pca_columns(gdf_test, P1_LISTS[2011])
    print(f"2011 P1: requested {len(P1_LISTS[2011])}, resolved {len(p1_cols)} columns")
    X, idx = prepare_pca_matrix(gdf_test, p1_cols)
    if X is not None:
        pca, scores, var = run_pca(X, n_components=3)
        print(f"  PCA shape {X.shape}, variance explained (first 3): {var}")
else:
    print("DATA_BERSIH gpkg not found; run cells after setting path.")

2011 P1: requested 127, resolved 73 columns


## Load each year's gpkg and run PCA (P1 and P2)


In [5]:
YEARS = [2011, 2014, 2018, 2019, 2020, 2021, 2024]
N_COMPONENTS = 1  
pca_results = []  # year, paper, n_vars, n_obs, var_explained, path

for year in YEARS:
    cand = [
        DATA_BERSIH / f"adm4_podes_{year}_with_iup_flag.gpkg",
        DATA_BERSIH / f"adm4_podes{year}_with_iup_flag.gpkg",
    ]
    gpkg_path = None
    for p in cand:
        if p.exists():
            gpkg_path = p
            break
    if gpkg_path is None:
        print(f"{year}: gpkg not found, skip")
        continue

    with warnings.catch_warnings(action="ignore", category=UserWarning):
        gdf = gpd.read_file(gpkg_path)

    for paper_name, var_list in [("P1", P1_LISTS.get(year, [])), ("P2", P2_LISTS.get(year, []))]:
        if not var_list:
            continue
        pca_cols = resolve_pca_columns(gdf, var_list)
        X, idx = prepare_pca_matrix(gdf, pca_cols)
        if X is None:
            print(f"{year} {paper_name}: no valid matrix (cols={len(pca_cols)}, skip)")
            pca_results.append({"year": year, "paper": paper_name, "n_vars": len(pca_cols), "n_obs": 0, "var_explained": None, "path": str(gpkg_path)})
            continue

        pca, scores, var_ratio = run_pca(X, n_components=N_COMPONENTS)
        if pca is None:
            print(f"{year} {paper_name}: PCA fit failed")
            continue

        col_score = f"pca_{paper_name}_pc1"
        gdf[col_score] = np.nan
        gdf.loc[idx, col_score] = scores[:, 0]

        var_str = str(round(float(var_ratio[0]), 4)) if len(var_ratio) else ""
        print(f"{year} {paper_name}: n_vars={X.shape[1]}, n_obs={X.shape[0]}, var_explained_pc1={var_str}")
        pca_results.append({
            "year": year,
            "paper": paper_name,
            "n_vars": X.shape[1],
            "n_obs": X.shape[0],
            "var_explained": float(var_ratio[0]) if len(var_ratio) else None,
            "path": str(gpkg_path),
        })

    out_path = gpkg_path.parent / (gpkg_path.stem + "_pca.gpkg")
    gdf.to_file(out_path, driver="GPKG", index=False)
    print(f"  {year}: saved {out_path.name}")
    if SAVE_CSV:
        csv_path = out_path.with_suffix(".csv")
        gdf.drop(columns=["geometry"], errors="ignore").to_csv(csv_path, index=False)
        print(f"  {year}: saved {csv_path.name}")

pd.DataFrame(pca_results).to_csv(DATA_BERSIH / "pca_summary.csv", index=False)
print("\nSummary saved to Data/DATA_BERSIH/pca_summary.csv")

2011 P1: n_vars=73, n_obs=18731, var_explained_pc1=0.161
2011 P2: n_vars=45, n_obs=18731, var_explained_pc1=0.163
  2011: saved adm4_podes_2011_with_iup_flag_pca.gpkg
  2011: saved adm4_podes_2011_with_iup_flag_pca.csv
2014 P1: n_vars=62, n_obs=79681, var_explained_pc1=0.1963
2014 P2: n_vars=54, n_obs=79681, var_explained_pc1=0.2403
  2014: saved adm4_podes_2014_with_iup_flag_pca.gpkg
  2014: saved adm4_podes_2014_with_iup_flag_pca.csv
2018 P1: n_vars=68, n_obs=80822, var_explained_pc1=0.2408
2018 P2: n_vars=53, n_obs=80822, var_explained_pc1=0.2396
  2018: saved adm4_podes_2018_with_iup_flag_pca.gpkg
  2018: saved adm4_podes_2018_with_iup_flag_pca.csv
2019 P1: n_vars=11, n_obs=80641, var_explained_pc1=0.3516
2019 P2: n_vars=46, n_obs=80641, var_explained_pc1=0.2603
  2019: saved adm4_podes_2019_with_iup_flag_pca.gpkg
  2019: saved adm4_podes_2019_with_iup_flag_pca.csv
2020 P1: n_vars=29, n_obs=80466, var_explained_pc1=0.3298
2020 P2: n_vars=46, n_obs=80466, var_explained_pc1=0.2574
  

## Smelter data and intervention (kecamatan-level)

- **Regimen 1 (campuran)**: Where a specific desa is listed → only that desa is intervention; where desa is missing → entire kecamatan is intervention (all desa in that kecamatan).
- **Regimen 2 (full kecamatan)**: All desa in any kecamatan that has a smelter are intervention (regardless of whether a specific desa is listed).

In [6]:
# Load smelter data and build intervention key sets
SMELTER_PATH = BASE / "Data" / "Data Smelter Indonesia.xlsx"
if not SMELTER_PATH.exists():
    raise FileNotFoundError(f"Smelter file not found: {SMELTER_PATH}")
smelter = pd.read_excel(SMELTER_PATH, sheet_name="Sheet1")
if smelter.empty:
    raise ValueError("Smelter sheet is empty.")

def norm(s):
    if pd.isna(s) or s is None:
        return ""
    return str(s).strip().upper()

smelter["_prov"] = smelter["provinsi"].map(norm)
smelter["_kab"] = smelter["kabupaten"].map(norm)
smelter["_kec"] = smelter["kecamatan"].map(norm)
smelter["_desa"] = smelter["desa"].map(norm)

# Regimen 2: all kecamatan with smelter -> all desa in that kec get flag
smelter_kec_set = set(zip(smelter["_prov"], smelter["_kab"], smelter["_kec"]))

# Regimen 1: (prov, kab, kec) where desa is missing -> whole kec; (prov, kab, kec, desa) where desa present -> desa itu saja
smelter_kec_only_set = set()
smelter_desa_set = set()
for _, r in smelter.iterrows():
    key_kec = (r["_prov"], r["_kab"], r["_kec"])
    if r["_desa"]:
        smelter_desa_set.add((r["_prov"], r["_kab"], r["_kec"], r["_desa"]))
    else:
        smelter_kec_only_set.add(key_kec)

print("Smelter rows:", len(smelter))
print("Reg2: kecamatan keys (all desa in kec)", len(smelter_kec_set))
print("Reg1: desa-level keys", len(smelter_desa_set), "| kec-only keys", len(smelter_kec_only_set))

Smelter rows: 23
Reg2: kecamatan keys (all desa in kec) 16
Reg1: desa-level keys 14 | kec-only keys 4


## Panel data

In [7]:
def norm(s):
    if pd.isna(s) or s is None: return ""
    return str(s).strip().upper()

def assign_smelter_flags(df, prov_col, kab_col, kec_col, desa_col,
                         smelter_kec_set, smelter_desa_set, smelter_kec_only_set):
    """Set smelter_reg1 and smelter_reg2 (0/1) in place. df must have prov, kab, kec, desa columns."""
    p = df[prov_col].map(norm)
    k = df[kab_col].map(norm)
    kec = df[kec_col].map(norm)
    d = df[desa_col].map(norm)
    keys_kec = list(zip(p, k, kec))
    keys_desa = list(zip(p, k, kec, d))
    df["smelter_reg2"] = [1 if x in smelter_kec_set else 0 for x in keys_kec]
    reg1 = []
    for i, x_desa in enumerate(keys_desa):
        x_kec = keys_kec[i]
        if x_desa in smelter_desa_set or x_kec in smelter_kec_only_set:
            reg1.append(1)
        else:
            reg1.append(0)
    df["smelter_reg1"] = reg1
    return df

In [8]:
# Build panel
ID_STABLE = "id_desa"
PROV_COL, KAB_COL, KEC_COL, DESA_COL = "NAMA_PROV", "NAMA_KAB", "NAMA_KEC", "NAMA_DESA"

list_dfs = []
for year in YEARS:
    p = DATA_BERSIH / f"adm4_podes_{year}_with_iup_flag_pca.gpkg"
    if not p.exists():
        continue
    with warnings.catch_warnings(action="ignore", category=UserWarning):
        g = gpd.read_file(p)
    prov = PROV_COL if PROV_COL in g.columns else "ADM1_EN"
    kab = KAB_COL if KAB_COL in g.columns else "ADM2_EN"
    kec = KEC_COL if KEC_COL in g.columns else "ADM3_EN"
    desa = DESA_COL if DESA_COL in g.columns else "ADM4_EN"
    if not all(c in g.columns for c in [prov, kab, kec, desa]):
        print(f"{year}: missing admin columns, skip")
        continue
    id_c = "id_desa_numeric" if "id_desa_numeric" in g.columns else ("ID_DESA" if "ID_DESA" in g.columns else "ADM4_PCODE")
    base_cols = [id_c, prov, kab, kec, desa, "pca_P1_pc1", "pca_P2_pc1", "has_iup"]
    base_cols = [c for c in base_cols if c in g.columns]
    sub = g[base_cols].copy()
    sub = sub.rename(columns={id_c: ID_STABLE, prov: PROV_COL, kab: KAB_COL, kec: KEC_COL, desa: DESA_COL})
    sub["year"] = year
    sub = assign_smelter_flags(sub, PROV_COL, KAB_COL, KEC_COL, DESA_COL,
                               smelter_kec_set, smelter_desa_set, smelter_kec_only_set)
    list_dfs.append(sub)

if not list_dfs:
    print("No panel data: no *_pca.gpkg found or all years skipped.")
else:
    panel_all = pd.concat(list_dfs, ignore_index=True)
    KEEP_COLS = [c for c in [ID_STABLE, "year", "NAMA_PROV", "NAMA_KAB", "NAMA_KEC", "NAMA_DESA",
                              "pca_P1_pc1", "pca_P2_pc1", "has_iup"] if c in panel_all.columns]

    panel_PCA_only = panel_all[KEEP_COLS].copy()
    out_pca_only = DATA_BERSIH / "panel_PCA_only.csv"
    panel_PCA_only.to_csv(out_pca_only, index=False)
    print("Saved panel_PCA_only:", out_pca_only, "| shape", panel_PCA_only.shape)

    panel_PCA_with_smelter = panel_all[KEEP_COLS + ["smelter_reg1", "smelter_reg2"]].copy()
    out_smelter = DATA_BERSIH / "panel_PCA_with_smelter.csv"
    panel_PCA_with_smelter.to_csv(out_smelter, index=False)
    print("Saved panel_PCA_with_smelter:", out_smelter, "| shape", panel_PCA_with_smelter.shape)

    panel_PCA_with_smelter_reg1 = panel_all[KEEP_COLS].copy()
    panel_PCA_with_smelter_reg1["smelter"] = panel_all["smelter_reg1"].values
    panel_PCA_with_smelter_reg1.to_csv(DATA_BERSIH / "panel_PCA_with_smelter_reg1.csv", index=False)
    panel_PCA_with_smelter_reg2 = panel_all[KEEP_COLS].copy()
    panel_PCA_with_smelter_reg2["smelter"] = panel_all["smelter_reg2"].values
    panel_PCA_with_smelter_reg2.to_csv(DATA_BERSIH / "panel_PCA_with_smelter_reg2.csv", index=False)
    print("Saved panel_PCA_with_smelter_reg1.csv and panel_PCA_with_smelter_reg2.csv")
    print("Smelter reg1 count:", panel_all["smelter_reg1"].sum(), "| reg2 count:", panel_all["smelter_reg2"].sum())

Saved panel_PCA_only: /Users/athamawardi/Desktop/Research-Projects/KRE_Equity/Data/DATA_BERSIH/panel_PCA_only.csv | shape (575334, 9)
Saved panel_PCA_with_smelter: /Users/athamawardi/Desktop/Research-Projects/KRE_Equity/Data/DATA_BERSIH/panel_PCA_with_smelter.csv | shape (575334, 11)
Saved panel_PCA_with_smelter_reg1.csv and panel_PCA_with_smelter_reg2.csv
Smelter reg1 count: 234 | reg2 count: 886


## variance explained (first PC) by year and paper

In [9]:
summary = pd.DataFrame(pca_results)
if len(summary) > 0:
    pivot = summary.pivot(index="year", columns="paper", values="var_explained")
    display(pivot)
    display(summary[["year", "paper", "n_vars", "n_obs", "var_explained"]])

paper,P1,P2
year,,
2011,0.160960,0.162967
2014,0.196283,0.240347
2018,0.240809,0.239619
2019,0.351626,0.260327
2020,0.329771,0.257356
2021,0.147201,0.213750
2024,0.148196,0.171895


,year,paper,n_vars,n_obs,var_explained
0,2011,P1,73,18731,0.160960
1,2011,P2,45,18731,0.162967
2,2014,P1,62,79681,0.196283
3,2014,P2,54,79681,0.240347
4,2018,P1,68,80822,0.240809
5,2018,P2,53,80822,0.239619
6,2019,P1,11,80641,0.351626
7,2019,P2,46,80641,0.260327
8,2020,P1,29,80466,0.329771
9,2020,P2,46,80466,0.257356
